# MLDL glaucoma project - single source of truth

This notebook is the canonical execution file. It performs acquisition, provenance checks, dataset inventory, sample-sheet generation, manifest validation, and controlled handoff updates. Training is deliberately gated until all required datasets are verified. No restricted dataset is downloaded or redistributed by this notebook.

## Operating rules

- ACRIMA has classification labels only; never report segmentation metrics for it.
- RIM-ONE V3 suspects are excluded from binary classification unless a protocol explicitly includes them.
- REFUGE and ORIGA candidate archives are not accepted as verified until their contents, labels, masks, and splits are inspected.
- DRISHTI-GS requires official access; place it locally only after registration and do not redistribute it.
- Test labels and masks are never used to create training crops.
- All reported metrics must come from saved per-image predictions on frozen splits.

In [ ]:
from pathlib import Path
import csv
import hashlib
import io
import json
import subprocess
import sys
import zipfile

PROJECT_ROOT = Path('/content/mldl_project')
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd()
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_AUDIT = PROJECT_ROOT / 'data' / 'audit'
OUTPUTS = PROJECT_ROOT / 'outputs'
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_AUDIT.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Raw data:', DATA_RAW)


In [ ]:
DATASETS = {
    'ACRIMA': {
        'archive': DATA_RAW / 'ACRIMA_Database.zip',
        'status': 'verified',
        'source': 'Figshare article 7613135, Database.zip',
        'labels': 'filename: _g means glaucoma; otherwise normal',
        'masks': 'none',
        'restriction': 'CC BY 4.0 source metadata; follow source terms',
    },
    'RIM-ONE V3': {
        'archive': DATA_RAW / 'RIM-ONE-r3.zip',
        'status': 'verified',
        'source': 'University of La Laguna original archive',
        'labels': 'filename prefix N/G/S',
        'masks': 'paired disc and cup average masks',
        'restriction': 'use according to source terms',
    },
    'REFUGE': {
        'archive': DATA_RAW / 'REFUGE_figshare.rar',
        'status': 'candidate_pending_inspection',
        'source': 'Figshare article 26049574, candidate archive',
        'labels': 'must be verified from archive',
        'masks': 'must be verified from archive',
        'restriction': 'do not assume challenge splits or encodings',
    },
    'ORIGA': {
        'archive': DATA_RAW / 'ORIGA_candidate_figshare.zip',
        'status': 'candidate_pending_inspection',
        'source': 'Figshare article 24549217, candidate archive',
        'labels': 'must be verified from archive',
        'masks': 'must be verified from archive',
        'restriction': 'do not assume this is the original ORIGA-light release',
    },
    'DRISHTI-GS': {
        'archive': None,
        'status': 'registration_required',
        'source': 'IIIT Hyderabad official access process',
        'labels': 'must be verified after authorized access',
        'masks': 'must be verified after authorized access',
        'restriction': 'never redistribute restricted images',
    },
}
Path(DATA_AUDIT / 'dataset_registry.json').write_text(json.dumps({k: {**v, 'archive': str(v['archive']) if v['archive'] else None} for k, v in DATASETS.items()}, indent=2), encoding='utf-8')
for name, record in DATASETS.items():
    print(f"{name}: {record['status']} | archive={record['archive']}")


## Acquisition

The repository downloader is resumable and records SHA-256 provenance. Large REFUGE and ORIGA downloads may take hours and can be run one at a time if the host throttles parallel transfers. Set `DOWNLOAD_PUBLIC_CANDIDATES = True` only when the storage and network are ready.

In [ ]:
DOWNLOAD_PUBLIC_CANDIDATES = False
PUBLIC_DOWNLOADS = ('acrima', 'rim_one_r3', 'refuge_candidate', 'origa_candidate')
if DOWNLOAD_PUBLIC_CANDIDATES:
    downloader = PROJECT_ROOT / 'tools' / 'download_verified.py'
    for dataset in PUBLIC_DOWNLOADS:
        subprocess.run([sys.executable, str(downloader), dataset, '--output', str(DATA_RAW)], cwd=PROJECT_ROOT, check=True)
else:
    print('Downloads are disabled. Existing complete archives and retained .part files will be inspected.')


In [ ]:
def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def archive_state(path):
    if path is None:
        return {'state': 'not_present', 'bytes': None, 'sha256': None}
    if path.exists():
        return {'state': 'complete_candidate', 'bytes': path.stat().st_size, 'sha256': sha256(path)}
    partial = Path(str(path) + '.part')
    if partial.exists():
        return {'state': 'partial', 'bytes': partial.stat().st_size, 'sha256': None}
    return {'state': 'not_present', 'bytes': None, 'sha256': None}

archive_inventory = {name: archive_state(record['archive']) for name, record in DATASETS.items()}
(DATA_AUDIT / 'archive_inventory.json').write_text(json.dumps(archive_inventory, indent=2), encoding='utf-8')
for name, state in archive_inventory.items():
    print(name, state)


In [ ]:
def zip_members(path):
    with zipfile.ZipFile(path) as archive:
        return archive.namelist()

def inspect_zip(path):
    members = zip_members(path)
    suffixes = {}
    for member in members:
        suffix = Path(member).suffix.lower() or '[none]'
        suffixes[suffix] = suffixes.get(suffix, 0) + 1
    return {'members': len(members), 'suffixes': suffixes, 'sample_members': members[:10]}

for name in ('ACRIMA', 'RIM-ONE V3', 'ORIGA'):
    path = DATASETS[name]['archive']
    if path and path.exists():
        try:
            print(name, inspect_zip(path))
        except zipfile.BadZipFile as exc:
            print(name, 'not a readable ZIP:', exc)
rar_path = DATASETS['REFUGE']['archive']
if rar_path and rar_path.exists():
    try:
        import rarfile
    except ImportError:
        print('REFUGE is RAR; install rarfile and an unrar backend before inspecting it.')
    else:
        with rarfile.RarFile(rar_path) as archive:
            print('REFUGE:', len(archive.infolist()), 'members')


In [ ]:
# Run the two completed, source-specific audits and preserve their manifests.
for script in ('audit_acrima.py', 'audit_rim_one_r3.py'):
    subprocess.run([sys.executable, str(PROJECT_ROOT / 'tools' / script)], cwd=PROJECT_ROOT, check=True)

required_manifests = (DATA_AUDIT / 'acrima_manifest.csv', DATA_AUDIT / 'rim_one_r3_manifest.csv')
assert all(path.exists() for path in required_manifests), 'Expected audit manifest was not produced.'
print('Verified manifests:', ', '.join(str(path) for path in required_manifests))


In [ ]:
from PIL import Image, ImageDraw

def make_acrima_sheet():
    archive_path = DATASETS['ACRIMA']['archive']
    if not archive_path.exists():
        print('ACRIMA archive unavailable; sample sheet skipped.')
        return
    with zipfile.ZipFile(archive_path) as archive:
        names = sorted(name for name in archive.namelist() if name.lower().endswith('.jpg'))
        chosen = [next(name for name in names if '_g_' not in Path(name).name.lower()), next(name for name in names if '_g_' in Path(name).name.lower())]
        sheet = Image.new('RGB', (900, 460), 'white')
        draw = ImageDraw.Draw(sheet)
        for index, name in enumerate(chosen):
            with Image.open(io.BytesIO(archive.read(name))) as image:
                image.thumbnail((410, 380))
                sheet.paste(image, (20 + index * 440, 20))
            draw.text((20 + index * 440, 415), Path(name).name, fill='black')
        output = DATA_AUDIT / 'acrima_samples.jpg'
        sheet.save(output, quality=92)
        print('Wrote', output)

make_acrima_sheet()
print('Existing RIM-ONE sample:', DATA_AUDIT / 'rim_sample.jpg')


In [ ]:
def csv_counts(path, key):
    with path.open(newline='', encoding='utf-8') as stream:
        rows = list(csv.DictReader(stream))
    counts = {}
    for row in rows:
        value = row[key]
        counts[value] = counts.get(value, 0) + 1
    return len(rows), counts

acrima_count, acrima_labels = csv_counts(DATA_AUDIT / 'acrima_manifest.csv', 'label')
rim_count, rim_labels = csv_counts(DATA_AUDIT / 'rim_one_r3_manifest.csv', 'label_original')
counts = {
    'ACRIMA': {'images': acrima_count, 'labels': acrima_labels, 'disc_masks': False, 'cup_masks': False, 'verification': 'passed'},
    'RIM-ONE V3': {'images': rim_count, 'labels': rim_labels, 'disc_masks': rim_count, 'cup_masks': rim_count, 'verification': 'passed'},
    'REFUGE': {'images': None, 'labels': None, 'disc_masks': None, 'cup_masks': None, 'verification': 'pending archive inspection'},
    'ORIGA': {'images': None, 'labels': None, 'disc_masks': None, 'cup_masks': None, 'verification': 'pending archive inspection'},
    'DRISHTI-GS': {'images': None, 'labels': None, 'disc_masks': None, 'cup_masks': None, 'verification': 'pending authorized access'},
}
(DATA_AUDIT / 'dataset_counts.json').write_text(json.dumps(counts, indent=2), encoding='utf-8')
print(json.dumps(counts, indent=2))


## Existing measured baselines

These are preserved outputs, not regenerated claims. The notebook only displays saved metrics and keeps them separate from the pending five-dataset experiment.

In [ ]:
for metrics_path in (OUTPUTS / 'acrima_resnet18_frozen' / 'metrics.json', OUTPUTS / 'rim_unet_baseline' / 'metrics.json'):
    if metrics_path.exists():
        print(metrics_path)
        print(metrics_path.read_text(encoding='utf-8'))
    else:
        print('Missing saved metrics:', metrics_path)


### Matched RIM-ONE whole-image vs disc-crop experiment

This is a matched binary experiment on the fixed 80/19/25 non-suspect split. Both variants use the same frozen ResNet-18 feature extractor and logistic regularization candidates. Training and validation crops use ground-truth disc masks. Test crops use the saved U-Net predicted disc mask, never test ground truth.

In [ ]:
RUN_RIM_WHOLE_VS_CROP = False
if RUN_RIM_WHOLE_VS_CROP:
    run_project_script('train_rim_whole_vs_crop.py')
    run_project_script('render_rim_crop_comparison.py')

crop_dir = OUTPUTS / 'rim_whole_vs_crop'
if (crop_dir / 'metrics.json').exists():
    crop_metrics = json.loads((crop_dir / 'metrics.json').read_text(encoding='utf-8'))
    print('Whole n/AUC/accuracy:', crop_metrics['n_test'], round(crop_metrics['whole']['auc'], 4), round(crop_metrics['whole']['accuracy'], 4))
    print('Crop n/AUC/accuracy:', crop_metrics['n_test'], round(crop_metrics['crop']['auc'], 4), round(crop_metrics['crop']['accuracy'], 4))
    display(Image.open(crop_dir / 'comparison.png'))
else:
    print('RIM whole/crop comparison artifacts missing.')

### Pooled segmentation baseline

A compact randomly initialized U-Net was trained across RIM-ONE r3, ORIGA, and REFUGE. The test set is 32 RIM-ONE, 325 ORIGA, and 800 REFUGE images. DRISHTI-GS remains absent, and this is not the final pretrained-encoder segmenter.

In [ ]:
RUN_POOLED_SEGMENTATION = False
if RUN_POOLED_SEGMENTATION:
    run_project_script('train_pooled_segmentation.py')
    run_project_script('render_pooled_segmentation.py')

pooled_seg_dir = OUTPUTS / 'pooled_segmentation'
if (pooled_seg_dir / 'metrics.json').exists():
    pooled_seg = json.loads((pooled_seg_dir / 'metrics.json').read_text(encoding='utf-8'))
    for record in pooled_seg['test_metrics']:
        print(record['dataset'], 'n=', record['n_test'], 'disc Dice/IoU=', round(record['test_disc_dice'],4), '/', round(record['test_disc_iou'],4), 'cup Dice/IoU=', round(record['test_cup_dice'],4), '/', round(record['test_cup_iou'],4))
    for panel in sorted((pooled_seg_dir / 'examples').glob('*.png')):
        display(Image.open(panel))
else:
    print('Pooled segmentation artifacts missing.')

In [ ]:
# Hard gate: no new final training until all five datasets have verified inventories.
RUN_FINAL_TRAINING = False
verified = {name for name, record in DATASETS.items() if record['status'] == 'verified' and record['archive'] and record['archive'].exists()}
if RUN_FINAL_TRAINING:
    required = set(DATASETS)
    missing = required - verified
    if missing:
        raise RuntimeError(f'Final training blocked; datasets not verified: {sorted(missing)}')
    print('All dataset gates passed. Add the approved training cells here.')
else:
    print('Final training remains disabled until Level 1 is complete:', sorted(set(DATASETS) - verified))


In [ ]:
handoff = PROJECT_ROOT / 'tools' / 'HANDOFF.md'
handoff_text = f'''# HANDOFF.md - Current Project State

**Last Updated**: 2026-11-18  
**Canonical workflow**: `PROJECT.ipynb`  
**Current Level**: Level 1 - Dataset Acquisition & Inventory  
**Status**: Partial verification complete; final five-dataset inventory is blocked on REFUGE, ORIGA, and authorized DRISHTI-GS access.

## Completed and verified
- ACRIMA archive and labels audited: 705 images, 396 glaucoma, 309 normal; no segmentation masks.
- RIM-ONE V3 archive audited: 85 normal, 39 glaucoma, 35 suspect; all 159 images have paired disc/cup average masks.
- Existing baseline outputs remain preserved under `outputs/` and are not presented as final five-dataset results.
- `PROJECT.ipynb` is the single execution entry point for acquisition, provenance, inventory, visual checks, and gates.

## Blockers and safeguards
- REFUGE and ORIGA candidate archives have retained partial downloads after remote timeout/reset errors; resume with the verified retrying downloader.
- REFUGE is a RAR archive and requires a compatible extraction backend before member-level inspection.
- ORIGA remains a candidate until its contents are verified against the intended release.
- DRISHTI-GS requires official registration/access and must not be redistributed.
- Do not start final model training, freeze final five-dataset splits, or publish cross-dataset metrics until all five inventories pass.

## Next exact actions
1. Run `PROJECT.ipynb` acquisition and archive inspection cells.
2. Resume REFUGE and ORIGA until complete, then verify archive members, labels, masks, and official splits.
3. Obtain DRISHTI-GS through the authorized process and audit it locally.
4. Complete counts and visual sample sheets for all five datasets.
5. Freeze manifests and train/validation/test splits; only then enable model training.
'''
handoff.write_text(handoff_text, encoding='utf-8')
print('Updated', handoff)
